# BridgeWatch AI: Model Training

Project: BridgeWatch AI, predicting whether a bridge's condition will deteriorate by its next inspection.
Author: Edmund Goldsberry, UMBC DATA 606 Capstone.

This notebook is being built step by step, one function at a time.

1. Assemble the training data (join the bridge dataset from eda.ipynb with the weather dataset).
2. Define the feature set and clean it up for modeling.
3. Split into train and test sets (time based, not random).
4. Build and fit a preprocessing and Logistic Regression pipeline.
5. Evaluate on the held out year.
6. Extract coefficients as an interpretability artifact.
7. Save the trained model for the Streamlit app.

Step 1 (this version): imports and data assembly.

## 1. Load and join the bridge and weather data

Two sources already exist from earlier work:

* `data/processed/bridge_deterioration_dataset.csv.gz`: one row per bridge per year to year transition
  (2020 to 21 through 2024 to 25), with the `deteriorated_next_period` label. A single gzip compressed file.
* `data/weather_annual_by_bridge_2020.csv` through `data/weather_annual_by_bridge_2025.csv`: one row per
  bridge per calendar year, with annual temperature and precipitation summaries from NOAA. Split into six
  plain csv files, one per year, so each file stays under GitHub's per file size limits.

They share a bridge ID (`STRUCTURE_NUMBER_008`), but the year column is named differently in each: the bridge
table calls it `from_year` (the year the features come from), the weather files call it `YEAR`. So the join
key is `(STRUCTURE_NUMBER_008, from_year)` matched to `(STRUCTURE_NUMBER_008, YEAR)`. This is the same join
already demonstrated in `eda.ipynb` Section 8.3.

The loaders below read directly from the GitHub repo (raw file URLs on the `main` branch), since files are
being pushed there as each step of this notebook is committed. If GitHub is missing a file, or there is no
network access, the loader falls back to a local copy instead.

In [1]:
import os
import urllib.error

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/goldiemonster/UMBC-DATA606-Capstone/main"
LOCAL_DATA_DIR = os.path.join("..", "data")
KEY = "STRUCTURE_NUMBER_008"
WEATHER_YEARS = range(2020, 2026)


def load_bridge_data():
    """Load the bridge-year-transition dataset built in eda.ipynb.

    Tries the GitHub repo first, and falls back to a local copy if that fails.
    """
    remote_path = "data/processed/bridge_deterioration_dataset.csv.gz"
    local_path = os.path.join(LOCAL_DATA_DIR, "processed", "bridge_deterioration_dataset.csv.gz")
    try:
        df = pd.read_csv(f"{GITHUB_RAW_BASE}/{remote_path}", compression="gzip")
        print(f"Loaded {remote_path} from GitHub")
        return df
    except (urllib.error.URLError, urllib.error.HTTPError, OSError) as exc:
        print(f"Could not load {remote_path} from GitHub ({exc}); falling back to {local_path}")
        return pd.read_csv(local_path, compression="gzip")


def load_weather_data():
    """Load the annual per-bridge weather dataset, split into one csv per year on GitHub.

    Tries to fetch all six yearly files from GitHub. If any single file is missing, the whole
    attempt is abandoned and a local, single gzip copy of the full dataset is used instead,
    so a partial GitHub fetch never gets silently mixed with local data.
    """
    try:
        frames = []
        for year in WEATHER_YEARS:
            remote_path = f"data/weather_annual_by_bridge_{year}.csv"
            frames.append(pd.read_csv(f"{GITHUB_RAW_BASE}/{remote_path}"))
        print(f"Loaded weather_annual_by_bridge_{{{min(WEATHER_YEARS)}..{max(WEATHER_YEARS)}}}.csv from GitHub")
        return pd.concat(frames, ignore_index=True)
    except (urllib.error.URLError, urllib.error.HTTPError, OSError) as exc:
        local_path = os.path.join(LOCAL_DATA_DIR, "processed", "weather_annual_by_bridge.csv.gz")
        print(f"Could not load all weather years from GitHub ({exc}); falling back to {local_path}")
        return pd.read_csv(local_path, compression="gzip")


def join_bridge_and_weather(bridge_df, weather_df):
    """Left join weather onto bridge_df, matching (KEY, from_year) to (KEY, YEAR)."""
    weather_renamed = weather_df.drop(columns="STATE").rename(columns={"YEAR": "weather_year"})
    merged = bridge_df.merge(
        weather_renamed,
        left_on=[KEY, "from_year"],
        right_on=[KEY, "weather_year"],
        how="left",
    ).drop(columns="weather_year")
    return merged

In [2]:
bridge_df = load_bridge_data()
weather_df = load_weather_data()

print(f"Bridge dataset: {bridge_df.shape[0]:,} rows x {bridge_df.shape[1]} columns")
print(f"Weather dataset: {weather_df.shape[0]:,} rows x {weather_df.shape[1]} columns")

df = join_bridge_and_weather(bridge_df, weather_df)

assert len(df) == len(bridge_df), "Join should not change the row count (weather is one row per bridge-year)"
weather_match_rate = df["tavg_mean_c"].notna().mean() * 100

print(f"\nJoined dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Weather match rate: {weather_match_rate:.1f}%")
df.head()

Loaded data/processed/bridge_deterioration_dataset.csv.gz from GitHub


Loaded weather_annual_by_bridge_{2020..2025}.csv from GitHub
Bridge dataset: 681,841 rows x 39 columns
Weather dataset: 841,608 rows x 8 columns



Joined dataset: 681,841 rows x 44 columns
Weather match rate: 98.5%


,STRUCTURE_NUMBER_008,COUNTY_CODE_003,LAT_016,LONG_017,OWNER_022,FUNCTIONAL_CLASS_026,YEAR_BUILT_027,TRAFFIC_LANES_ON_028A,ADT_029,YEAR_ADT_030,STRUCTURE_KIND_043A,STRUCTURE_TYPE_043B,MAIN_UNIT_SPANS_045,APPR_SPANS_046,MAX_SPAN_LEN_MT_048,STRUCTURE_LEN_MT_049,DECK_WIDTH_MT_052,DECK_COND_058,SUPERSTRUCTURE_COND_059,SUBSTRUCTURE_COND_060,CULVERT_COND_062,DATE_OF_INSPECT_090,INSPECT_FREQ_MONTHS_091,DECK_STRUCTURE_TYPE_107,PERCENT_ADT_TRUCK_109,SCOUR_CRITICAL_113,BRIDGE_CONDITION,LOWEST_RATING,STATE,YEAR,LOWEST_RATING_next,BRIDGE_CONDITION_next,from_year,to_year,deteriorated_next_period,age,lat,lon,months_since_inspection,tmax_mean_c,tmin_mean_c,tavg_mean_c,prcp_total_mm,freeze_thaw_days
0,06 0021,89,40454199.0,122197000.0,69,1,1941,4,19500,2009.0,3,9,8,7,192.0,1093.6,18.9,5,7,7,N,518,24,1,29.0,8,F,5,CA,2020,5,F,2020,2021,0,79,40.761664,-122.336111,24,25.256404,9.759584,17.508389,682.585938,3.0
1,19P0005,61,39000000.0,120562375.0,69,9,1935,1,75,2017.0,3,10,1,3,39.6,67.7,4.0,5,6,3,N,518,24,8,NaN,2,P,3,CA,2020,3,P,2020,2021,0,85,39.000000,-120.939931,24,23.319907,8.978441,16.149441,653.156250,17.0
2,1CA0070,111,34060810.0,119061488.0,73,9,1949,2,200,2013.0,1,4,1,0,10.4,10.4,17.2,6,5,6,N,819,24,2,2.0,5,F,5,CA,2020,5,F,2020,2021,0,71,34.102250,-119.104133,24,20.630550,12.516244,16.572938,167.789062,0.0
3,1CA0095,73,32433594.0,117084854.0,73,19,1985,2,100,2015.0,2,2,5,0,18.9,60.8,10.7,6,7,8,N,819,24,1,20.0,N,F,6,CA,2020,6,F,2020,2021,0,35,32.726650,-117.146817,24,22.369002,13.666090,18.017525,195.328125,0.0
4,1CA0141,111,34065929.0,119054787.0,73,9,1948,2,200,2013.0,1,5,4,0,6.9,27.4,15.9,7,7,5,N,819,24,2,2.0,5,F,5,CA,2020,5,F,2020,2021,0,72,34.116469,-119.096631,24,20.630550,12.516244,16.572938,167.789062,0.0
